# 🎓 NB-11: Knowledge Distillation – Claude as Teacher
**Goal:** Distill Claude's response knowledge into a tiny LSTM student model using soft targets from a BERT teacher.

**Modality:** Knowledge Distillation | **Models:** BERT (teacher) + BiLSTM (student)

In [ ]:
!pip install transformers torch datasets -q

In [ ]:
import json, re, random
random.seed(42)

def load_dataset(path="claude-opus-4_5-250x_jsonl.txt"):
    """Load Claude thinking dataset from JSONL."""
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def extract_fields(records):
    data = []
    for r in records:
        msgs = r["messages"]
        user = next((m["content"] for m in msgs if m["role"]=="user"), "")
        asst = next((m["content"] for m in msgs if m["role"]=="assistant"), "")
        think = re.findall(r"<think>(.*?)</think>", asst, re.DOTALL)
        response = re.sub(r"<think>.*?</think>", "", asst, flags=re.DOTALL).strip()
        data.append({
            "user": user,
            "assistant_full": asst,
            "thinking": " ".join(think).strip(),
            "response": response
        })
    return data

records = load_dataset()
data = extract_fields(records)
print(f"Loaded {len(data)} examples")
print(f"Sample user: {data[0]['user'][:80]}")
print(f"Sample think: {data[0]['thinking'][:120]}...")
print(f"Sample resp: {data[0]['response'][:120]}...")


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from transformers import BertTokenizer, BertModel
from torch.utils.data import Dataset as TDS, DataLoader

tok = BertTokenizer.from_pretrained("bert-base-uncased")

class TeacherBERT(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.head = nn.Linear(768, 256)
    def forward(self, **kw): return self.head(self.bert(**kw).pooler_output)

class StudentLSTM(nn.Module):
    def __init__(self, vocab=30522, emb=64, hid=128):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb)
        self.lstm = nn.LSTM(emb, hid, batch_first=True, bidirectional=True)
        self.head = nn.Linear(hid*2, 256)
    def forward(self, ids):
        return self.head(self.lstm(self.emb(ids))[0][:,0,:])

teacher = TeacherBERT()
student = StudentLSTM()
device = "cuda" if torch.cuda.is_available() else "cpu"
teacher.to(device); student.to(device)

texts = [f"{d['user']} {d['response'][:200]}" for d in data]
enc = tok(texts, truncation=True, padding="max_length", max_length=128, return_tensors="pt")

class DistDS(TDS):
    def __len__(self): return len(texts)
    def __getitem__(self, i): return {k: v[i] for k,v in enc.items()}

opt = torch.optim.AdamW(student.parameters(), lr=1e-3)
loader = DataLoader(DistDS(), batch_size=16, shuffle=True)

for epoch in range(5):
    total = 0
    for batch in loader:
        batch = {k: v.to(device) for k,v in batch.items()}
        with torch.no_grad():
            t_out = teacher(**batch)
        s_out = student(batch["input_ids"])
        loss = F.mse_loss(s_out, t_out)
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item()
    print(f"Epoch {epoch+1} Distillation Loss: {total/len(loader):.4f}")
print("✅ Student model distilled from BERT teacher!")
